In [3]:
import sys, os
sys.path.append(os.path.abspath(".."))

import torch
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

In [4]:
pt_paths = [
    # r"../data/spectral/truthfulQA_Judgelabels_and_eigs_top10.pt",
    r"../data/spectral/triviaQA_Judgelabels_and_eigs_top10.pt",
    r"../data/spectral/mmlu_Judgelabels_and_eigs_top10.pt",
    r"../data/spectral/math_Judgelabels_and_eigs_top10.pt"
]

rows = []
for path in pt_paths:
    payload = torch.load(path, map_location="cpu")
    for sample_id, item in payload["data"].items():
        rows.append({
            "id": sample_id,
            "label": item.get("label", None),
            "domain": item.get("domain", None),
        })

df_labels = pd.DataFrame(rows).drop_duplicates(subset=["id"])
df_labels["label"] = df_labels["label"].astype(str).str.lower()

df_labels.head()

,id,label,domain
0,triviaqa_00000_t0.1_ans00,hallucination,Physical Sciences
1,triviaqa_00001_t0.1_ans00,hallucination,Arts & Literature
2,triviaqa_00002_t0.1_ans00,hallucination,General Knowledge
3,triviaqa_00003_t0.1_ans00,hallucination,Arts & Literature
4,triviaqa_00004_t0.1_ans00,hallucination,Philosophy & Religion


In [5]:
data = torch.load("../data/processed/df_pca.pt", map_location="cpu")

X = data["X"].numpy()
ids = data["id"]
datasets = data["dataset"]

# rebuild dataframe
pc_cols = [f"PC{i+1}" for i in range(X.shape[1])]

df_pca = pd.DataFrame(X, columns=pc_cols)
df_pca.insert(0, "id", ids)
df_pca.insert(1, "dataset", datasets)

df_pca.head()

,id,dataset,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,...,PC1369,PC1370,PC1371,PC1372,PC1373,PC1374,PC1375,PC1376,PC1377,PC1378
0,triviaqa_00000_t0.1_ans00,truthfulqa,-44.182053,11.572464,-0.158500,-8.428147,-12.760209,-0.573039,-3.082951,-2.810615,...,-0.261632,-0.694015,-0.619766,0.251993,-0.325119,0.089933,-0.042068,-0.821198,-0.704396,-0.365106
1,triviaqa_00001_t0.1_ans00,truthfulqa,-47.176624,15.367263,-0.839482,-12.471917,-1.784634,2.181762,13.712010,2.802632,...,-0.008218,0.000905,-0.748834,-0.105519,-0.464503,-0.305665,-0.763447,0.206863,-0.022236,-0.264161
2,triviaqa_00002_t0.1_ans00,truthfulqa,-44.161472,11.508937,10.504965,-1.909707,2.297472,0.642891,11.306277,-5.342796,...,0.048174,0.432347,0.107536,0.282098,-0.489400,-1.177175,0.571867,0.756571,0.298008,-0.371687
3,triviaqa_00003_t0.1_ans00,truthfulqa,-38.254707,16.724350,5.560056,4.334621,-2.047990,-1.049951,-4.873804,0.018275,...,-0.599921,0.486538,-0.425430,-0.350028,0.252028,0.711712,0.052531,0.146374,-0.299138,0.118096
4,triviaqa_00004_t0.1_ans00,truthfulqa,-45.817905,12.770360,-13.607718,-6.582739,-5.345684,8.146999,-3.381339,-1.895386,...,-0.064618,-0.038639,-0.014426,0.069018,0.539320,0.281209,0.443333,-0.029841,0.102833,0.510708


In [6]:
df_pca_labeled = df_pca.merge(df_labels, on="id", how="left")
df_pca_labeled["label"].value_counts(dropna=False)

df_pca_labeled.head()

,id,dataset,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,...,PC1371,PC1372,PC1373,PC1374,PC1375,PC1376,PC1377,PC1378,label,domain
0,triviaqa_00000_t0.1_ans00,truthfulqa,-44.182053,11.572464,-0.158500,-8.428147,-12.760209,-0.573039,-3.082951,-2.810615,...,-0.619766,0.251993,-0.325119,0.089933,-0.042068,-0.821198,-0.704396,-0.365106,hallucination,Physical Sciences
1,triviaqa_00001_t0.1_ans00,truthfulqa,-47.176624,15.367263,-0.839482,-12.471917,-1.784634,2.181762,13.712010,2.802632,...,-0.748834,-0.105519,-0.464503,-0.305665,-0.763447,0.206863,-0.022236,-0.264161,hallucination,Arts & Literature
2,triviaqa_00002_t0.1_ans00,truthfulqa,-44.161472,11.508937,10.504965,-1.909707,2.297472,0.642891,11.306277,-5.342796,...,0.107536,0.282098,-0.489400,-1.177175,0.571867,0.756571,0.298008,-0.371687,hallucination,General Knowledge
3,triviaqa_00003_t0.1_ans00,truthfulqa,-38.254707,16.724350,5.560056,4.334621,-2.047990,-1.049951,-4.873804,0.018275,...,-0.425430,-0.350028,0.252028,0.711712,0.052531,0.146374,-0.299138,0.118096,hallucination,Arts & Literature
4,triviaqa_00004_t0.1_ans00,truthfulqa,-45.817905,12.770360,-13.607718,-6.582739,-5.345684,8.146999,-3.381339,-1.895386,...,-0.014426,0.069018,0.539320,0.281209,0.443333,-0.029841,0.102833,0.510708,hallucination,Philosophy & Religion


In [7]:
df_pca_labeled["label"].unique()

array(['hallucination', 'correct', 'refused', 'illogical'], dtype=object)

In [11]:
df_pca_labeled.value_counts("label", dropna=False)

label
correct          2719
hallucination    1655
illogical        1576
refused            50
Name: count, dtype: int64

In [8]:
# df_pca_labeled should have columns: PC1..PCn and a label column with:
# ['hallucination', 'correct', 'refused', 'illogical']

LABEL_COL = "label"  # <-- change if your column name is different

# -------------------------
# 1) Keep only the 4 classes (drop NaN/other)
# -------------------------
classes = ["hallucination", "correct", "refused", "illogical"]

df_train = df_pca_labeled.copy()
df_train[LABEL_COL] = df_train[LABEL_COL].astype(str).str.lower()
df_train = df_train[df_train[LABEL_COL].isin(classes)].dropna(subset=[LABEL_COL])

# -------------------------
# 2) Build X (PC columns) and y (4-class labels)
# -------------------------
pc_cols = [c for c in df_train.columns if c.startswith("PC")]
X = df_train[pc_cols].to_numpy()
y = df_train[LABEL_COL].to_numpy()

print("X shape:", X.shape)
print("y distribution:\n", pd.Series(y).value_counts())

X shape: (6000, 1378)
y distribution:
 correct          2719
hallucination    1655
illogical        1576
refused            50
Name: count, dtype: int64


In [9]:
# -------------------------
# 3) Train/test split (stratified)
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=33,
    stratify=y
)

# -------------------------
# 4) Multi-class classifier
#    (Logistic regression supports multinomial)
# -------------------------
log_clf = Pipeline([
    ("scaler", StandardScaler()),  # PCs are already scaled-ish, but this is safe
    ("lr", LogisticRegression(
        max_iter=5000,
        class_weight="balanced",   # helps if classes are imbalanced
        multi_class="multinomial",
        solver="lbfgs",
        # l1_ratio=0.5,              # elastic net mixing
    ))
])

log_clf.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('lr',
                 LogisticRegression(class_weight='balanced', max_iter=5000,
                                    multi_class='multinomial'))])

In [10]:
# -------------------------
# 5) Evaluate
# -------------------------
y_pred = log_clf.predict(X_test)

print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=3))

print("\nConfusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(confusion_matrix(y_test, y_pred, labels=classes),
                   index=[f"true_{c}" for c in classes],
                   columns=[f"pred_{c}" for c in classes]))


Classification report:
               precision    recall  f1-score   support

      correct      0.592     0.579     0.586       544
hallucination      0.418     0.492     0.452       331
    illogical      0.818     0.711     0.761       315
      refused      0.000     0.000     0.000        10

     accuracy                          0.585      1200
    macro avg      0.457     0.446     0.450      1200
 weighted avg      0.598     0.585     0.590      1200


Confusion matrix (rows=true, cols=pred):
                    pred_hallucination  pred_correct  pred_refused  \
true_hallucination                 163           152             2   
true_correct                       196           315             0   
true_refused                         2             5             0   
true_illogical                      29            60             2   

                    pred_illogical  
true_hallucination              14  
true_correct                    33  
true_refused                 